# DegreeDistribution

Notebook for analyzing distributions and descriptive statistics on network metrics:
- indegree
- outdegree
- instrength
- total degree

Required inputs:
- interactions.parquet
- users.parquet

Output:
- Overview table with mean, median, standard deviation, percentiles, etc.
- figures

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "axes.titlesize": 16,
    "axes.labelsize": 21,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "axes.grid": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

BASE_DIR = Path('.')
INTERACTIONS_PATH = BASE_DIR / 'interactions.parquet'
USERS_PATH = BASE_DIR / 'users.parquet'
OUT_DIR = BASE_DIR / 'figures' / 'distribuzione_gradi'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input interactions: {INTERACTIONS_PATH.resolve()}')
print(f'Input users: {USERS_PATH.resolve()}')
print(f'Output figure dir: {OUT_DIR.resolve()}')

In [ ]:
# Column-pruned reading: we load only the useful columns.
interactions = pd.read_parquet(INTERACTIONS_PATH, columns=['source', 'target'])
users = pd.read_parquet(USERS_PATH, columns=['day', 'user_id'])


interactions = interactions.dropna(subset=['source', 'target']).copy()
interactions['source'] = interactions['source'].astype('string')
interactions['target'] = interactions['target'].astype('string')

users = users.dropna(subset=['day', 'user_id']).copy()
users['day'] = pd.to_datetime(users['day'], errors='coerce')
users['user_id'] = users['user_id'].astype('string')
users = users.dropna(subset=['day', 'user_id'])

print(f'interactions: {len(interactions):,}')
print(f'activeusers: {len(users):,}')
print(f'unique users: {users["user_id"].nunique(dropna=True):,}')

In [ ]:
def summarize_array(a: np.ndarray) -> dict[str, float]:
    a = np.asarray(a)
    if a.size == 0:
        return {
            'count': 0,
            'mean': np.nan,
            'median': np.nan,
            'std': np.nan,
            'min': np.nan,
            'p5': np.nan,
            'p25': np.nan,
            'p75': np.nan,
            'p95': np.nan,
            'max': np.nan,
        }

    q5, q25, q75, q95 = np.percentile(a, [5, 25, 75, 95])
    return {
        'count': int(a.size),
        'mean': float(np.mean(a)),
        'median': float(np.median(a)),
        'std': float(np.std(a)),
        'min': float(np.min(a)),
        'p5': float(q5),
        'p25': float(q25),
        'p75': float(q75),
        'p95': float(q95),
        'max': float(np.max(a)),
    }


def _make_log_bins(arr: np.ndarray, n_bins: int = 80) -> np.ndarray:
    a = np.asarray(arr)
    if a.size == 0:
        return np.array([1.0, 2.0])

    xmin = float(np.min(a))
    xmax = float(np.max(a))

    if xmin <= 0:
        xmin = 1.0

    if xmin == xmax:
        xmax = xmin * 1.2

    return np.logspace(np.log10(xmin), np.log10(xmax), n_bins)


def save_metric_plots(
    values: np.ndarray,
    xlabel: str,
    basename: str,
    out_dir: Path,
    color: str,
    bins: int = 80,
) -> list[dict[str, str]]:
    arr = np.asarray(values)
    arr = arr[arr >= 1]  # k >= 1: exclude zeros

    if arr.size == 0:
        return []

    log_bins = _make_log_bins(arr, n_bins=bins)
    outputs: list[dict[str, str]] = []

    # 1) Frequency with logarithmic bins
    fig, ax = plt.subplots(figsize=(8.8, 5.2), facecolor='white')
    ax.set_facecolor('white')
    ax.hist(arr, bins=log_bins, color=color, alpha=0.9, edgecolor='black', linewidth=0.35)
    ax.set_xscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Frequency')

    png_path = out_dir / f'{basename}_freq_logbins.png'
    pdf_path = out_dir / f'{basename}_freq_logbins.pdf'
    fig.tight_layout()
    fig.savefig(png_path, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    outputs.append({'variant': 'freq_logbins', 'png': str(png_path), 'pdf': str(pdf_path)})

    # 2) Density with log-log axes
    fig, ax = plt.subplots(figsize=(8.8, 5.2), facecolor='white')
    ax.set_facecolor('white')
    ax.hist(
        arr,
        bins=log_bins,
        density=True,
        color=color,
        alpha=0.88,
        edgecolor='black',
        linewidth=0.35,
    )
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')

    png_path = out_dir / f'{basename}_density_loglog.png'
    pdf_path = out_dir / f'{basename}_density_loglog.pdf'
    fig.tight_layout()
    fig.savefig(png_path, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    outputs.append({'variant': 'density_loglog', 'png': str(png_path), 'pdf': str(pdf_path)})

    return outputs

In [ ]:
# Node universe: union of users from users and endpoints present in interactions.
node_index = pd.Index(users['user_id'].dropna().unique())
node_index = node_index.union(pd.Index(interactions['source'].dropna().unique()))
node_index = node_index.union(pd.Index(interactions['target'].dropna().unique()))
n_nodes = len(node_index)

# Vector encoding → integer indices for calculations with np.bincount.
src_codes = pd.Categorical(interactions['source'], categories=node_index).codes
tgt_codes = pd.Categorical(interactions['target'], categories=node_index).codes

valid = (src_codes >= 0) & (tgt_codes >= 0)
src = src_codes[valid].astype(np.int64, copy=False)
tgt = tgt_codes[valid].astype(np.int64, copy=False)

# In-strength/out-strength over all interactions, with weights.
out_strength = np.bincount(src, minlength=n_nodes)
in_strength = np.bincount(tgt, minlength=n_nodes)


edges_df = pd.DataFrame({'s': src, 't': tgt}, copy=False).drop_duplicates(ignore_index=True)
s_unique = edges_df['s'].to_numpy(dtype=np.int64, copy=False)
t_unique = edges_df['t'].to_numpy(dtype=np.int64, copy=False)

out_degree = np.bincount(s_unique, minlength=n_nodes)
in_degree = np.bincount(t_unique, minlength=n_nodes)


u = np.minimum(s_unique, t_unique)
v = np.maximum(s_unique, t_unique)
und_df = pd.DataFrame({'u': u, 'v': v}, copy=False).drop_duplicates(ignore_index=True)
u_und = und_df['u'].to_numpy(dtype=np.int64, copy=False)
v_und = und_df['v'].to_numpy(dtype=np.int64, copy=False)

degree_total = np.bincount(np.concatenate([u_und, v_und]), minlength=n_nodes)
loops_mask = (u_und == v_und)
if loops_mask.any():
    degree_total[u_und[loops_mask]] -= 1

print(f'total nodes: {n_nodes:,}')
print(f'unique directed edges: {len(edges_df):,}')
print(f'unique undirected edges: {len(und_df):,}')

In [ ]:
network_metrics = {
    'indegree': in_degree,
    'outdegree': out_degree,
    'instrength': in_strength,
    'degree_total': degree_total,
}

plot_labels = {
    'indegree': 'In-degree',
    'outdegree': 'Out-degree',
    'instrength': 'In-strength',
    'degree_total': 'Total degree',
}

# Distinct colors for overlay plot
plot_colors = {
    'indegree': '#0072B2',
    'instrength': '#D55E00',
    'outdegree': '#009E73',
    'degree_total': '#9467bd',
}

network_summary_rows = []
network_fig_rows = []


def _smooth_array(x: np.ndarray, sigma_bins: float = 1.5) -> np.ndarray:
    # Gaussian smoothing in index space
    if x.size == 0:
        return x
    sigma = float(max(0.5, sigma_bins))
    half_width = int(max(1, np.ceil(3 * sigma)))
    idx = np.arange(-half_width, half_width + 1)
    kernel = np.exp(-0.5 * (idx / sigma) ** 2)
    kernel = kernel / kernel.sum()
    return np.convolve(x, kernel, mode='same')


def save_overlay_density_plot(metrics: dict, names: list[str], xlabel: str, basename: str, out_dir: Path, bins: int = 80):
    outputs: list[dict[str, str]] = []
    fig, ax = plt.subplots(figsize=(8.8, 5.2), facecolor='white')
    ax.set_facecolor('white')

    markers = {
    'indegree': 'o',      
    'instrength': '^',   
    'outdegree': 's',  }  


    for name in names:
      values = np.asarray(metrics[name])
      arr = values[values >= 1]

      if arr.size == 0:
          continue

      log_bins = _make_log_bins(arr, n_bins=bins)
      counts, edges = np.histogram(arr, bins=log_bins, density=True)
      centers = np.sqrt(edges[:-1] * edges[1:])
      smooth = _smooth_array(counts, sigma_bins=1.5)

      ax.plot(
          centers,
          smooth,
          label=plot_labels.get(name, name),
          color=plot_colors.get(name, None),
          linestyle='None',
          marker=markers.get(name, 'o'),
          markersize=6,
          alpha=0.95,)
          

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.legend(loc='best')

    png_path = out_dir / f'{basename}.png'
    pdf_path = out_dir / f'{basename}.pdf'
    fig.tight_layout()
    fig.savefig(png_path, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    plt.close(fig)

    outputs.append({'variant': basename, 'png': str(png_path), 'pdf': str(pdf_path)})
    return outputs

# Creates the overlaid plot for indegree, instrength, and outdegree.
overlay_names = ['indegree', 'instrength', 'outdegree']
overlay_outputs = save_overlay_density_plot(
    network_metrics,
    overlay_names,
    xlabel='Degree/Strength',
    basename='overlay_indegree_instrength_outdegree_density_loglog',
    out_dir=OUT_DIR,
    bins=80,
)

for out in overlay_outputs:
    network_fig_rows.append({'metric': 'overlay', 'variant': out['variant'], 'png': out['png'], 'pdf': out['pdf']})



for name, values in network_metrics.items():
    stats = summarize_array(values)
    stats['metric'] = name
    network_summary_rows.append(stats)

    saved_plots = save_metric_plots(
        values=values,
        xlabel=plot_labels[name],
        basename=f'hist_{name}',
        out_dir=OUT_DIR,
        color=plot_colors.get(name, '#2ca02c'),
        bins=80,
    )

    for plot_info in saved_plots:
        network_fig_rows.append(
            {
                'metric': name,
                'variant': plot_info['variant'],
                'png': plot_info['png'],
                'pdf': plot_info['pdf'],
            }
        )

network_summary_df = pd.DataFrame(network_summary_rows)[
    ['metric', 'count', 'mean', 'median', 'std', 'min', 'p5', 'p25', 'p75', 'p95', 'max']
]
network_fig_df = pd.DataFrame(network_fig_rows)

display(network_summary_df)
display(network_fig_df)

In [ ]:
# Export opzionale della panoramica rete in CSV
network_summary_csv = OUT_DIR / 'network_summary.csv'
network_summary_df.to_csv(network_summary_csv, index=False)

print(f'Salvato: {network_summary_csv.resolve()}')

saved = sorted(OUT_DIR.glob('*'))
pd.DataFrame({'file': [str(p) for p in saved]}).head(50)